# Notebook 02 — Train CNN Models

**Experiments covered here:**

| Exp | Description | Course concept |
|-----|-------------|----------------|
| 1   | Train custom 3-layer CNN from scratch | Conv2D, BN, ReLU, MaxPool (Lec 5, 7) |
| 3   | Compare vs ResNet-18 (transfer learning) | Fine-tuning (Lec 9) |
| 6   | Ablation — batch size effect on accuracy | BN + gradient noise (Lec 6, 7) |

**Expected accuracy:** 85–95 % on held-out synthetic test set.


In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import matplotlib.pyplot as plt
import numpy as np

from config import *
from src.training.dataset  import get_dataloaders, CLASS_TO_IDX, SHORT_NAMES
from src.models.cnn        import PatternCNN
from src.models.resnet18   import get_resnet18, count_trainable
from src.training.trainer  import Trainer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load dataset

In [ ]:
train_loader, val_loader = get_dataloaders(
    data_dir    = DATA_DIR,
    batch_size  = BATCH_SIZE,
    img_size    = IMG_SIZE,
    num_workers = NUM_WORKERS,
)

# Quick sanity check
imgs, labels = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}  Labels: {labels[:8].tolist()}')

## Experiment 1 — Custom 3-layer PatternCNN

In [ ]:
cnn_model = PatternCNN(num_classes=7, dropout_p=DROPOUT_P)
print(f'PatternCNN trainable parameters: {cnn_model.count_parameters():,}')

cnn_trainer = Trainer(
    cnn_model, train_loader, val_loader,
    lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    device=DEVICE, step_size=STEP_SIZE, gamma=GAMMA,
)
cnn_save = os.path.join(ROOT, 'checkpoints', 'cnn_best.pt')
cnn_history = cnn_trainer.fit(n_epochs=N_EPOCHS, patience=PATIENCE, save_path=cnn_save)

## Experiment 3 — ResNet-18 Transfer Learning

In [ ]:
resnet = get_resnet18(num_classes=7, pretrained=True, frozen_backbone=False)
print(f'ResNet-18 trainable parameters: {count_trainable(resnet):,}')

resnet_trainer = Trainer(
    resnet, train_loader, val_loader,
    lr=5e-4, weight_decay=WEIGHT_DECAY,          # lower LR for pretrained backbone
    device=DEVICE, step_size=STEP_SIZE, gamma=GAMMA,
)
resnet_save = os.path.join(ROOT, 'checkpoints', 'resnet18_best.pt')
resnet_history = resnet_trainer.fit(n_epochs=N_EPOCHS, patience=PATIENCE, save_path=resnet_save)

## Training curves

In [ ]:
def plot_history(history, title, ax_loss, ax_acc):
    epochs = range(1, len(history['train_loss']) + 1)
    ax_loss.plot(epochs, history['train_loss'], label='Train Loss')
    ax_loss.plot(epochs, history['val_loss'],   label='Val Loss', linestyle='--')
    ax_loss.set_title(f'{title} — Loss')
    ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Cross-Entropy Loss')
    ax_loss.legend()

    ax_acc.plot(epochs, history['train_acc'], label='Train Acc')
    ax_acc.plot(epochs, history['val_acc'],   label='Val Acc',   linestyle='--')
    ax_acc.set_title(f'{title} — Accuracy')
    ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy')
    ax_acc.legend()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
plot_history(cnn_history,    'PatternCNN',  axes[0, 0], axes[0, 1])
plot_history(resnet_history, 'ResNet-18',   axes[1, 0], axes[1, 1])
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'figures', 'training_curves.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest Val Acc — PatternCNN : {max(cnn_history['val_acc']):.4f}")
print(f"Best Val Acc — ResNet-18  : {max(resnet_history['val_acc']):.4f}")

## Confusion matrix — PatternCNN

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

results = cnn_trainer.evaluate_full(val_loader, class_names=CLASS_NAMES)
print('PatternCNN Val Classification Report:')
print(results['report'])

cm = results['confusion_matrix']
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=SHORT_NAMES, yticklabels=SHORT_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — PatternCNN (validation set)')
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'figures', 'confusion_matrix_cnn.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## Experiment 6 — Batch-size ablation (Lecture 6 & 7)

Batch size affects:
- **BatchNorm quality** — smaller batches → noisier mean/variance estimates (Lec 7 p19–21)
- **Gradient noise** — smaller batches → noisier gradient estimates (Lec 6 p6–7)

In [ ]:
batch_results = {}

for bs in [16, 32, 64, 128]:
    print(f'\n--- Batch size {bs} ---')
    tr_l, vl_l = get_dataloaders(DATA_DIR, batch_size=bs, img_size=IMG_SIZE,
                                  num_workers=NUM_WORKERS)
    m = PatternCNN(num_classes=7, dropout_p=DROPOUT_P)
    t = Trainer(m, tr_l, vl_l, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
                device=DEVICE, step_size=STEP_SIZE, gamma=GAMMA)
    h = t.fit(n_epochs=15, patience=5)  # shorter run for ablation
    batch_results[bs] = max(h['val_acc'])
    print(f'  Best val acc: {batch_results[bs]:.4f}')

print('\nBatch size ablation summary:')
for bs, acc in batch_results.items():
    print(f'  bs={bs:4d}  val_acc={acc:.4f}')

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(list(batch_results.keys()), list(batch_results.values()),
         marker='o', linewidth=2)
plt.xlabel('Batch size')
plt.ylabel('Best val accuracy')
plt.title('Batch-size ablation (PatternCNN, 15 epochs)')
plt.xticks(list(batch_results.keys()))
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'results', 'figures', 'batch_size_ablation.png'),
            dpi=150, bbox_inches='tight')
plt.show()